In [15]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import fdrcorrection

# ========== 1. ЗАГРУЗКА ДАННЫХ ==========
df = pd.read_csv('PSI_download_GBM.txt', sep='\t', low_memory=False)

# ========== 2. УДАЛЕНИЕ КЛИНИЧЕСКИХ СТРОК ==========
clinical_keywords = ['Gender', 'Age', 'Height_cm', 'Weight_kg', 'Tumor_Status', 
                     'Vital_Status', 'Pathologic_Tumor_Stage', 'Pathologic_N',
                     'Pathologic_T', 'Absolute_Purity_Value', 
                     'Tissue_Specific_Attribute_1', 'Tissue_Specific_Attribute_2',
                     'Tissue_Specific_Attribute_3']
df = df[~df['symbol'].isin(clinical_keywords)]

# ========== 3. ВАШ СПИСОК ГЕНОВ (КЛАСТЕР 2) ==========
cluster2_genes = [
    "BCL2L11", "HNRNPK", "MAPK8", "BCL2", "H3-3B", "CXCR4", "CD44", "CDH1", "RHOA",
    "CCND2", "CDK6", "MAP2K1", "HRAS", "CDKN2A", "BRAF", "CCNE1", "TP53", "EGFR",
    "HNRNPA1", "MYCN", "KIT", "AKT3", "IRS1", "ALK", "SOX2", "ERCC1", "CASP9", "MLH1",
    "ERBB4", "SMAD3", "FGFR1", "HSP90AA1", "CDK4", "PMS2", "FGFR3", "FGFR2", "CASP8",
    "MTOR", "SRSF2", "MCL1", "NRAS", "POLD1", "PTEN", "MMP9", "SRSF3", "SRSF1", "SRSF9",
    "BCL2L1", "SRSF6", "SRSF7", "AKT1", "FOXO1", "MAPK1", "SRSF5", "TRA2B", "HNRNPH1",
    "AKT2", "PTBP1", "IGF1"
]

# ========== 4. ОСТАВЛЯЕМ ТОЛЬКО СТРОКИ С НУЖНЫМИ ГЕНАМИ ==========
df_cluster = df[df['symbol'].isin(cluster2_genes)].copy()
print(f"Найдено событий для генов кластера 2: {len(df_cluster)}")

# ========== 5. ОПРЕДЕЛЯЕМ ОБРАЗЦЫ ==========
sample_cols = [c for c in df.columns if c.startswith('TCGA_')]
tumor_cols = [c for c in sample_cols if not c.endswith('_Norm')]
normal_cols = [c for c in sample_cols if c.endswith('_Norm')]

print(f"Опухолевых образцов: {len(tumor_cols)}")
print(f"Нормальных образцов: {len(normal_cols)}")

# ========== 6. ДИФФЕРЕНЦИАЛЬНЫЙ АНАЛИЗ (MANN‑WHITNEY) ==========
results = []
for idx, row in df_cluster.iterrows():
    tumor_psi = row[tumor_cols].astype(float).dropna()
    normal_psi = row[normal_cols].astype(float).dropna()
    if len(tumor_psi) >= 3 and len(normal_psi) >= 3:
        stat, pval = mannwhitneyu(tumor_psi, normal_psi, alternative='two-sided')
        delta = tumor_psi.mean() - normal_psi.mean()
        results.append({
            'gene': row['symbol'],
            'event_type': row['splice_type'],
            'exons': row['exons'],
            'delta_PSI': delta,                # ← здесь имя колонки: delta_PSI
            'p_value': pval,
            'mean_tumor': tumor_psi.mean(),
            'mean_normal': normal_psi.mean(),
            'n_tumor': len(tumor_psi),
            'n_normal': len(normal_psi)
        })

df_results = pd.DataFrame(results)

# ========== 7. ПОПРАВКА НА МНОЖЕСТВЕННЫЕ СРАВНЕНИЯ (FDR) ==========
if len(df_results) > 0:
    df_results['p_adj'] = fdrcorrection(df_results['p_value'].values)[1]
else:
    df_results['p_adj'] = np.nan

# ========== 8. СОРТИРОВКА ПО АБСОЛЮТНОЙ delta_PSI ==========
# Убеждаемся, что колонка 'delta_PSI' существует
if 'delta_PSI' not in df_results.columns:
    raise KeyError("Колонка 'delta_PSI' не найдена. Доступны: " + str(df_results.columns.tolist()))

df_results = df_results.sort_values('delta_PSI', key=abs, ascending=False)

# ========== 9. ВЫВОД ТОП-10 СОБЫТИЙ ==========
print("\nТОП-10 СОБЫТИЙ ПО |ΔPSI| (без фильтра значимости):")
top10 = df_results.head(10)
print(top10[['gene', 'event_type', 'exons', 'delta_PSI', 'p_value', 'p_adj']].to_string())

# ========== 10. ФИЛЬТРАЦИЯ ЗНАЧИМЫХ (ПОРОГИ МОЖНО МЕНЯТЬ) ==========
delta_threshold = 0.15   # можно уменьшить до 0.1
p_threshold = 0.1        # можно уменьшить до 0.05

sig_df = df_results[(abs(df_results['delta_PSI']) > delta_threshold) & (df_results['p_adj'] < p_threshold)]
print(f"\nЗначимых событий (|ΔPSI|>{delta_threshold}, p_adj<{p_threshold}): {len(sig_df)}")

if len(sig_df) > 0:
    print(sig_df[['gene', 'event_type', 'exons', 'delta_PSI', 'p_adj']].head(10).to_string())
else:
    print("Нет значимых событий при текущих порогах.")

# ========== 11. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ ==========
df_results.to_csv('cluster2_all_events.csv', index=False)
sig_df.to_csv('cluster2_significant_events.csv', index=False)
print("\nРезультаты сохранены в файлы: cluster2_all_events.csv и cluster2_significant_events.csv")

Найдено событий для генов кластера 2: 157
Опухолевых образцов: 155
Нормальных образцов: 5

ТОП-10 СОБЫТИЙ ПО |ΔPSI| (без фильтра значимости):
        gene event_type              exons  delta_PSI   p_value     p_adj
148    MAPK8         ME                7|8  -0.544586  0.000463  0.003656
0       MTOR         AP               39.1   0.294960  0.001154  0.006658
30      IGF1         AT                  6  -0.289690  0.001644  0.008502
63      BCL2         RI                1.2  -0.256239  0.005539  0.022457
140   HNRNPK         AP                  1   0.219433  0.000147  0.002476
72     ERCC1         AP                2.1   0.213505  0.001211  0.006729
11     FGFR2         AT               28.2   0.213340  0.000202  0.002476
149  HNRNPA1         ES  7.1:7.2:8:9.1:9.2   0.189214  0.000264  0.002476
108     RHOA         ES              3:4:5  -0.188510  0.000235  0.002476
90     CASP8         AT                 14   0.188163  0.000009  0.001275

Значимых событий (|ΔPSI|>0.15, p_adj<0.1): 